# Qwen3.5 9B + 4B — Colab GPU Endpoint

Run the cells **from top to bottom** on a Colab GPU runtime (Tesla T4 or better).

This notebook downloads/reuses both GGUF models, builds `llama.cpp` with CUDA if needed, starts the two model servers, adds a small authenticated routing proxy, opens one ngrok tunnel, and tests both endpoints.

**Before running:** In Colab, select **Runtime → Change runtime type → GPU**. Your runtime must stay connected while the endpoints are in use. If the runtime resets, rerun the notebook from the top.

**Security:** The old ngrok token and model API key were included in the previous notebook/conversation. Revoke the exposed ngrok token in the ngrok dashboard and create a new one. This notebook prompts for the new token and a new API key without displaying them. Do not share a notebook after saving secrets or outputs.


## 1. Check GPU

In [ ]:
!nvidia-smi


## 2. Install Python dependencies

In [ ]:
%pip -q install -U "huggingface_hub>=0.30" pyngrok fastapi uvicorn httpx requests


## 3. Configure secrets

In [ ]:
import getpass
import os

# Prefer Colab Secrets (key icon in the left sidebar):
#   NGROK_AUTHTOKEN = your NEW ngrok token
#   MODEL_API_KEY = a strong random secret
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    NGROK_TOKEN = None

if not NGROK_TOKEN:
    NGROK_TOKEN = getpass.getpass("Paste your NEW ngrok authtoken (input hidden): ").strip()

try:
    from google.colab import userdata
    MODEL_API_KEY = userdata.get("MODEL_API_KEY")
except Exception:
    MODEL_API_KEY = None

if not MODEL_API_KEY:
    MODEL_API_KEY = getpass.getpass("Enter a strong model API key (input hidden): ").strip()

if not NGROK_TOKEN or not MODEL_API_KEY:
    raise ValueError("Both ngrok token and model API key are required.")
if len(MODEL_API_KEY) < 20:
    raise ValueError("Use a stronger API key of at least 20 characters.")

print("Secrets loaded. Values are hidden.")


## 4. Download or reuse the models

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import os

MODEL_9B = hf_hub_download(
    repo_id="unsloth/Qwen3.5-9B-GGUF",
    filename="Qwen3.5-9B-Q4_K_M.gguf",
)
MODEL_4B = hf_hub_download(
    repo_id="unsloth/Qwen3.5-4B-GGUF",
    filename="Qwen3.5-4B-Q4_K_M.gguf",
)

for label, path in [("9B", MODEL_9B), ("4B", MODEL_4B)]:
    print(f"{label}: {path}")
    print(f"  Exists: {Path(path).is_file()} | Size: {Path(path).stat().st_size / (1024**3):.2f} GiB")


## 5. Build llama.cpp with CUDA (only if needed)

In [ ]:
import os, subprocess
from pathlib import Path

LLAMA_DIR = Path("/content/llama.cpp")
SERVER_BIN = LLAMA_DIR / "build/bin/llama-server"

if not LLAMA_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(LLAMA_DIR)],
        check=True,
    )

if not SERVER_BIN.exists():
    subprocess.run([
        "cmake", "-S", str(LLAMA_DIR), "-B", str(LLAMA_DIR / "build"),
        "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release",
    ], check=True)
    subprocess.run([
        "cmake", "--build", str(LLAMA_DIR / "build"),
        "--config", "Release", "-j", "2", "--target", "llama-server",
    ], check=True)

if not SERVER_BIN.exists():
    raise RuntimeError(f"llama-server binary not found: {SERVER_BIN}")
print("llama-server ready:", SERVER_BIN)


## 6. Start both model servers safely

In [ ]:
import subprocess, time, requests, os
from pathlib import Path

LOG_9B = Path("/content/qwen9b.log")
LOG_4B = Path("/content/qwen4b.log")

def health(port):
    try:
        r = requests.get(f"http://127.0.0.1:{port}/health", timeout=3)
        return r.status_code == 200, f"{r.status_code} {r.text}"
    except requests.RequestException as e:
        return False, str(e)

def start_server_if_needed(port, model_path, alias, ctx_size, log_path):
    # Reuse a healthy server if the cell is rerun.
    ok, detail = health(port)
    if ok:
        print(f"Port {port} already healthy; reusing it.")
        return

    # Do not start a second process if a server is already listening/loading.
    try:
        r = requests.get(f"http://127.0.0.1:{port}/health", timeout=3)
        if r.status_code == 503 and "Loading model" in r.text:
            print(f"Port {port} is still loading; will wait.")
        else:
            print(f"Port {port} responded {r.status_code}; checking readiness.")
    except requests.RequestException:
        pass

    # Only start if port is not currently listening.
    probe = subprocess.run(
        ["bash", "-lc", f"ss -ltn '( sport = :{port} )' | tail -n +2"],
        capture_output=True, text=True,
    ).stdout.strip()
    if not probe:
        log_handle = open(log_path, "a")
        cmd = [
            str(SERVER_BIN), "-m", str(model_path),
            "--host", "0.0.0.0", "--port", str(port),
            "--n-gpu-layers", "99", "--ctx-size", str(ctx_size),
            "--alias", alias, "--api-key", MODEL_API_KEY,
        ]
        proc = subprocess.Popen(
            cmd, cwd="/content", stdout=log_handle, stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        print(f"Started {alias} on port {port}; PID {proc.pid}; log {log_path}")
    else:
        print(f"Port {port} is already listening; not starting a duplicate process.")

def wait_healthy(port, label, timeout_s=300):
    deadline = time.time() + timeout_s
    last = "not checked"
    while time.time() < deadline:
        ok, last = health(port)
        if ok:
            print(f"{label} ready: {last}")
            return
        time.sleep(5)
    log_path = LOG_9B if port == 8000 else LOG_4B
    print(log_path.read_text(errors="replace")[-5000:] if log_path.exists() else "No log file.")
    raise TimeoutError(f"{label} did not become healthy within {timeout_s}s. Last status: {last}")

start_server_if_needed(8000, MODEL_9B, "qwen3.5-9b-q4km", 2048, LOG_9B)
wait_healthy(8000, "Qwen 3.5 9B", timeout_s=300)

start_server_if_needed(8001, MODEL_4B, "qwen3.5-4b-q4km", 2048, LOG_4B)
wait_healthy(8001, "Qwen 3.5 4B", timeout_s=300)

!nvidia-smi


## 7. Start the authenticated routing proxy

In [ ]:
from fastapi import FastAPI, Request, Response
import httpx, threading, uvicorn, time, requests

app = FastAPI()
PORTS = {"9b": 8000, "4b": 8001}

@app.api_route("/{model}/{path:path}", methods=["GET", "POST", "PUT", "PATCH", "DELETE", "OPTIONS"])
async def route_request(model: str, path: str, request: Request):
    if model not in PORTS:
        return Response("Use /9b/ or /4b/", status_code=404)

    # Require authentication for model API routes. Health checks remain public.
    if path.startswith("v1/"):
        expected = f"Bearer {MODEL_API_KEY}"
        if request.headers.get("authorization") != expected:
            return Response('{"error":"Unauthorized"}', status_code=401, media_type="application/json")

    headers = {
        k: v for k, v in request.headers.items()
        if k.lower() not in ("host", "content-length", "connection")
    }
    try:
        async with httpx.AsyncClient() as client:
            upstream = await client.request(
                request.method,
                f"http://127.0.0.1:{PORTS[model]}/{path}",
                params=request.url.query,
                content=await request.body(),
                headers=headers,
                timeout=180,
            )
        excluded = ("content-length", "transfer-encoding", "connection", "content-encoding")
        response_headers = {k: v for k, v in upstream.headers.items() if k.lower() not in excluded}
        return Response(
            content=upstream.content,
            status_code=upstream.status_code,
            headers=response_headers,
        )
    except httpx.RequestError as e:
        return Response(f"Upstream model server unavailable: {e}", status_code=502)

# Reuse an existing proxy thread if this cell has already been run.
proxy_ok = False
try:
    proxy_ok = requests.get("http://127.0.0.1:9002/9b/health", timeout=2).status_code == 200
except requests.RequestException:
    pass

if proxy_ok:
    print("Proxy on port 9002 is already responding; reusing it.")
else:
    proxy_config = uvicorn.Config(app, host="0.0.0.0", port=9002, log_level="info")
    proxy_server = uvicorn.Server(proxy_config)
    proxy_thread = threading.Thread(target=proxy_server.run, daemon=True)
    proxy_thread.start()
    for _ in range(30):
        try:
            r9 = requests.get("http://127.0.0.1:9002/9b/health", timeout=2)
            r4 = requests.get("http://127.0.0.1:9002/4b/health", timeout=2)
            if r9.status_code == 200 and r4.status_code == 200:
                break
        except requests.RequestException:
            pass
        time.sleep(1)
    print("Proxy health:", r9.status_code, r9.text, "|", r4.status_code, r4.text)


## 8. Open ngrok tunnel to the proxy

In [ ]:
from pyngrok import ngrok
import time

ngrok.set_auth_token(NGROK_TOKEN)

# Reuse a tunnel to port 9002 if one already exists; otherwise create one.
proxy_url = None
for existing in ngrok.get_tunnels():
    addr = str(existing.config.get("addr", ""))
    if addr.endswith(":9002") or addr == "9002":
        proxy_url = existing.public_url
        print("Reusing existing tunnel:", proxy_url)
        break

if proxy_url is None:
    tunnel = ngrok.connect(addr=9002, proto="http")
    proxy_url = tunnel.public_url

print("Public proxy URL:", proxy_url)
print("Qwen 3.5 9B chat endpoint:", proxy_url + "/9b/v1/chat/completions")
print("Qwen 3.5 4B chat endpoint:", proxy_url + "/4b/v1/chat/completions")
print("9B health:", proxy_url + "/9b/health")
print("4B health:", proxy_url + "/4b/health")


## 9. Verify health and run a real inference test

In [ ]:
import requests

for route in ("/9b/health", "/4b/health"):
    r = requests.get(proxy_url + route, timeout=15)
    print(route, r.status_code, r.text)
    r.raise_for_status()

for name, route in [
    ("Qwen 3.5 9B", "/9b/v1/chat/completions"),
    ("Qwen 3.5 4B", "/4b/v1/chat/completions"),
]:
    print(f"\nTesting {name}...")
    r = requests.post(
        proxy_url + route,
        headers={
            "Authorization": f"Bearer {MODEL_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": "local-model",
            "messages": [{"role": "user", "content": "Reply with exactly: Hello from the model!"}],
            "temperature": 0,
            "max_tokens": 30,
        },
        timeout=180,
    )
    print("HTTP", r.status_code)
    print(r.text[:1200])
    r.raise_for_status()


## Using these endpoints from your Planner

Use the same public base URL printed in Cell 8, with distinct model paths:

- 9B base URL: `https://YOUR-NGROK-HOST/9b`
- 4B base URL: `https://YOUR-NGROK-HOST/4b`

For an OpenAI-compatible client, the full chat completion paths are `/v1/chat/completions` after the respective base URL. Set the API key to the same secret you entered in Cell 3. Keep the Colab runtime connected while your Planner uses the endpoints.

If a model fails to start, inspect `/content/qwen9b.log` or `/content/qwen4b.log`. Don't run `ngrok.kill()` as a routine step; it stops all ngrok tunnels.
